In [1]:
!pip install -U ecmwf-opendata xarray cfgrib netCDF4 pandas eccodes zarr

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 2.6 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.6 MB 2.6 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.6 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.6 MB 2.5 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/9.6 MB 2.5 MB/s eta 0:00:03
   -------------- ------------------------- 3.4/9.6 MB 2.4 MB/s eta 0:00:03
   ---------------- ----------------------- 3.9/9.6 MB 2.5 MB/s eta 0:00:03
   ------------------ --------------------- 4.5/9.6 MB 2.4 MB/s eta 0:00:03
   -------------------- ------------------- 5.0/9.6 MB 2.4 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.6 MB 2.5 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.6 MB 2.5 MB/s eta 0:00:02
   -----------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.6 which is incompatible.


In [2]:
import os
import xarray as xr
import pandas as pd
from pathlib import Path

In [7]:
# ==============================
# USER CONFIGURATION
# ==============================

DATE = "20260919"
TIME = 0
STEP = 0

LATITUDE = (5, 35)
LONGITUDE = (65, 100)

OUTPUT_DIR = Path("ecmwf_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [8]:
# ==============================
# ECMWF FEATURE MAPPING
# ==============================

FEATURE_MAP = {
    "temperature": {
        "shortName": "2t",
        "name": "2 metre temperature"
    },
    "pressure": {
        "shortName": "msl",
        "name": "Mean Sea Level Pressure"
    }
}

print("Feature mapping created successfully!")

Feature mapping created successfully!


In [9]:
from ecmwf.opendata import Client

client = Client(
    source="ecmwf"
)

print("ECMWF client initialized successfully!")

ECMWF client initialized successfully!


In [10]:
# ==============================
# DOWNLOAD ECMWF DATA
# ==============================

grib_file = OUTPUT_DIR / "ecmwf_data.grib2"

client.retrieve(
    type="fc",
    stream="oper",
    levtype="sfc",
    param=["2t", "msl"],
    time=TIME,
    step=STEP,
    date=DATE,
    target=str(grib_file)
)

print("ECMWF data downloaded successfully!")
print("File:", grib_file)

20260919000000-0h-oper-fc.grib2:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.
ECMWF data downloaded successfully!
File: ecmwf_output\ecmwf_data.grib2


In [11]:
print("GRIB file exists:", grib_file.exists())
print("File:", grib_file)

GRIB file exists: True
File: ecmwf_output\ecmwf_data.grib2


In [12]:
# ==============================
# READ TEMPERATURE
# ==============================

temperature = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "2t"
        }
    }
)

print("Temperature dataset opened successfully!")
print(temperature)

Ignoring index file 'ecmwf_output\\ecmwf_data.grib2.47d85.idx' older than GRIB file


Temperature dataset opened successfully!
<xarray.Dataset> Size: 4MB
Dimensions:            (latitude: 721, longitude: 1440)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    t2m                (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-20T00:25 GRIB to CDM+CF via cfgrib-0.9.1...


In [13]:
# ==============================
# READ PRESSURE
# ==============================

pressure = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "msl"
        }
    }
)

print("Pressure dataset opened successfully!")
print(pressure)

Ignoring index file 'ecmwf_output\\ecmwf_data.grib2.47d85.idx' older than GRIB file


Pressure dataset opened successfully!
<xarray.Dataset> Size: 4MB
Dimensions:     (latitude: 721, longitude: 1440)
Coordinates:
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    meanSea     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    msl         (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-20T00:25 GRIB to CDM+CF via cfgrib-0.9.1...


In [14]:
# ==============================
# CROP TEMPERATURE
# ==============================

temperature = temperature.sel(
    latitude=slice(LATITUDE[1], LATITUDE[0]),
    longitude=slice(LONGITUDE[0], LONGITUDE[1])
)

print("Temperature region cropped!")
print(temperature)

Temperature region cropped!
<xarray.Dataset> Size: 70kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    t2m                (latitude, longitude) float32 68kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-20T00:25 GRIB to CDM+CF via cfgrib-0.9.1...


In [15]:
# ==============================
# CROP PRESSURE
# ==============================

pressure = pressure.sel(
    latitude=slice(LATITUDE[1], LATITUDE[0]),
    longitude=slice(LONGITUDE[0], LONGITUDE[1])
)

print("Pressure region cropped!")
print(pressure)

Pressure region cropped!
<xarray.Dataset> Size: 70kB
Dimensions:     (latitude: 121, longitude: 141)
Coordinates:
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 1kB 65.0 65.25 65.5 ... 99.5 99.75 100.0
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    meanSea     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    msl         (latitude, longitude) float32 68kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-20T00:25 GRIB to CDM+CF via cfgrib-0.9.1...


In [16]:
# ==============================
# UNIT CONVERSION
# ==============================

temperature_c = temperature["t2m"] - 273.15

pressure_hpa = pressure["msl"] / 100

print("Temperature converted: Kelvin → Celsius")
print("Pressure converted: Pa → hPa")

Temperature converted: Kelvin → Celsius
Pressure converted: Pa → hPa


In [17]:
print("Temperature range:")
print(
    float(temperature_c.min()),
    "to",
    float(temperature_c.max()),
    "°C"
)

print("\nPressure range:")
print(
    float(pressure_hpa.min()),
    "to",
    float(pressure_hpa.max()),
    "hPa"
)

Temperature range:
-18.809738159179688 to 30.940277099609375 °C

Pressure range:
1006.708740234375 to 1031.268798828125 hPa


In [18]:
# ==============================
# FINAL XARRAY DATASET
# ==============================

final_dataset = xr.Dataset(
    {
        "temperature_2m_C": temperature_c,
        "pressure_msl_hPa": pressure_hpa
    }
)

print("Final dataset created successfully!")
print(final_dataset)

Final dataset created successfully!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B 2026-09-19
    step               timedelta64[ns] 8B 00:00:00
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B 2026-09-19
    meanSea            float64 8B ...
Data variables:
    temperature_2m_C   (latitude, longitude) float32 68kB 7.347 7.44 ... 28.13
    pressure_msl_hPa   (latitude, longitude) float32 68kB 1.021e+03 ... 1.012...


In [19]:
# ==============================
# SAVE AS ZARR
# ==============================

zarr_store = OUTPUT_DIR / "ecmwf_weather_data.zarr"

final_dataset.to_zarr(
    zarr_store,
    mode="w"
)

print("Zarr dataset saved successfully!")
print("Path:", zarr_store)

Zarr dataset saved successfully!
Path: ecmwf_output\ecmwf_weather_data.zarr


C:\Users\adars\anaconda3\Lib\site-packages\zarr\api\asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [20]:
# ==============================
# OPEN ZARR
# ==============================

ds = xr.open_zarr(zarr_store)

print("Zarr dataset opened successfully!")
print(ds)

Zarr dataset opened successfully!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    pressure_msl_hPa   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>
    temperature_2m_C   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>


In [21]:
# ==============================
# ZARR VALIDATION
# ==============================

print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(ds.coords)

print("\nVariables:")
print(ds.data_vars)

Dimensions:
FrozenMappingWarningOnValuesAccess({'latitude': 121, 'longitude': 141})

Coordinates:
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...

Variables:
Data variables:
    pressure_msl_hPa  (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>
    temperature_2m_C  (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>


In [22]:
# ==============================
# DATA PREVIEW
# ==============================

print("Temperature:")
print(ds["temperature_2m_C"])

print("\nPressure:")
print(ds["pressure_msl_hPa"])

Temperature:
<xarray.DataArray 'temperature_2m_C' (latitude: 121, longitude: 141)> Size: 68kB
dask.array<open_dataset-temperature_2m_C, shape=(121, 141), dtype=float32, chunksize=(121, 141), chunktype=numpy.ndarray>
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...
Attributes: (12/30)
    GRIB_paramId:                             167
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         heightAboveGround
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_na